# CP2 paired verification — BSARec vanilla vs time-aware (Beauty + ML-1M)

Notebook ini menjalankan baseline BSARec vanilla dan BSARec + time-bucket embedding pada data, source commit, runtime, split, dan seed yang sama. Setiap pasangan memakai seed yang sama. Training tidak berjalan otomatis saat notebook dibuka.

**Sebelum mulai:** pilih GPU runtime Colab dan unggah empat file pada sel upload. Run mencakup enam baseline dan enam varian time-aware (dua dataset × tiga seed), jadi dapat memakan waktu. Jalankan sel satu per satu secara berurutan. Nama log/checkpoint baru memakai prefix `BSARec_V_CP2_` dan `BSARec_T_CP2_` agar tidak menimpa artefak CP1 atau pilot sebelumnya.

Bucket waktu hanya dipakai pada posisi histori; target bucket tidak dimasukkan. Timestamp Beauty/ML-1M ditafsirkan sesuai timestamp sumbernya. Evaluasi test dipakai sekali setelah checkpoint dipilih dari validasi.


In [1]:
# Setup: clone source yang dipin ke commit audit, lalu pasang dependensi non-Torch.
%cd /content
!git clone https://github.com/yehjin-shin/BSARec.git
%cd /content/BSARec
!git checkout c80bdc0f1a889b78fe9a27e2058389cc84caea84
%cd /content/BSARec/src
!pip -q install scipy tqdm
import torch
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('PERINGATAN: training akan berjalan di CPU dan kemungkinan sangat lambat.')


/content
Cloning into 'BSARec'...
remote: Enumerating objects: 592, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 592 (delta 28), reused 20 (delta 20), pack-reused 549 (from 1)
Receiving objects: 100% (592/592), 859.10 MiB | 26.56 MiB/s, done.
Resolving deltas: 100% (136/136), done.
Updating files: 100% (384/384), done.
/content/BSARec
Note: switching to 'c80bdc0f1a889b78fe9a27e2058389cc84caea84'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now a

In [2]:
# Upload langsung dari komputer ke sesi Colab; Google Drive tidak diperlukan.
from google.colab import files
from pathlib import Path
import re
import shutil

root = Path('/content/BSARec/src')
data_dir = root / 'data'
data_dir.mkdir(parents=True, exist_ok=True)
required = {
    'Beauty.txt': 'Beauty.txt',
    'Beauty_cp2_time_buckets.txt': 'Beauty_cp2_time_buckets.txt',
    'ML-1M.txt': 'ML-1M.txt',
    'ML-1M_cp2_time_buckets.txt': 'ML-1M_cp2_time_buckets.txt',
}

# Cek file yang mungkin sudah diunggah ke /content. Colab dapat menambahkan (1) pada nama duplikat.
def find_existing(expected):
    p = Path(expected)
    pattern = re.compile(re.escape(p.stem) + r'(?: \(\d+\))?' + re.escape(p.suffix))
    for folder in [Path('/content'), root]:
        if folder.exists():
            matches = [x for x in folder.iterdir() if x.is_file() and pattern.fullmatch(x.name)]
            if matches:
                return matches[-1]
    return None

existing = {name: find_existing(name) for name in required}
missing = [name for name, path in existing.items() if path is None]
if missing:
    print('Pilih file yang belum ditemukan:', missing)
    uploaded = files.upload()
    for expected in missing:
        p = Path(expected)
        pattern = re.compile(re.escape(p.stem) + r'(?: \(\d+\))?' + re.escape(p.suffix))
        candidates = [name for name in uploaded if pattern.fullmatch(Path(name).name)]
        if not candidates:
            raise FileNotFoundError(f'Belum diunggah: {expected}. File terunggah: {list(uploaded)}')
        # Upload payload disimpan langsung, aman walau namanya memiliki akhiran (1).
        (data_dir / expected).write_bytes(uploaded[candidates[-1]])
        print(f'{candidates[-1]} -> {data_dir / expected}')

for expected, source in existing.items():
    if source is not None:
        shutil.copy2(source, data_dir / expected)
        print(f'{source} -> {data_dir / expected}')

for expected in required:
    target = data_dir / expected
    if not target.exists() or target.stat().st_size == 0:
        raise FileNotFoundError(f'File input tidak tersedia atau kosong: {target}')
print('Empat file input siap di', data_dir)


/content/Beauty.txt -> /content/BSARec/src/data/Beauty.txt
/content/Beauty_cp2_time_buckets.txt -> /content/BSARec/src/data/Beauty_cp2_time_buckets.txt
/content/ML-1M.txt -> /content/BSARec/src/data/ML-1M.txt
/content/ML-1M_cp2_time_buckets.txt -> /content/BSARec/src/data/ML-1M_cp2_time_buckets.txt
Empat file input siap di /content/BSARec/src/data


## A. Baseline vanilla — jalankan sebelum patch time-aware

Sel berikut memakai source upstream tanpa patch waktu. Jalankan semua enam run ini lebih dulu. Parameter lain memakai default upstream yang sama dengan CP1; log akan mencatat Namespace lengkap.


In [ ]:
# Baseline vanilla Beauty, seed 42
!python main.py --data_name Beauty --model_type BSARec --lr 0.0005 --alpha 0.7 --c 5 --num_attention_heads 1 --seed 42 --train_name BSARec_V_CP2_Beauty_s42 --num_workers 4

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:431: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
2026-11-20 13:11:23,573 - Namespace(data_dir='./data/', output_dir='output/', data_name='Beauty', do_eval=False, load_model=None, train_name='BSARec_V_CP2_Beauty_s42', num_items=10, num_users=22364, lr=0.0005, batch_size=256, epochs=200, no_cuda=False, log_freq=1, patience=10, num_workers=4, seed=42, weight_decay=0.0, adam_beta1=0.9, adam_beta2=0.999, gpu_id='0', variance=5, model_type='BSARec', max_seq_length=50, hidden_size=64, num_hidden_layers=2, hidden_act='gelu', num_attention_heads=1, attention_probs_dropout_

In [ ]:
# Baseline vanilla Beauty, seed 43
!python main.py --data_name Beauty --model_type BSARec --lr 0.0005 --alpha 0.7 --c 5 --num_attention_heads 1 --seed 43 --train_name BSARec_V_CP2_Beauty_s43 --num_workers 4

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:431: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
2026-11-20 13:33:43,485 - Namespace(data_dir='./data/', output_dir='output/', data_name='Beauty', do_eval=False, load_model=None, train_name='BSARec_V_CP2_Beauty_s43', num_items=10, num_users=22364, lr=0.0005, batch_size=256, epochs=200, no_cuda=False, log_freq=1, patience=10, num_workers=4, seed=43, weight_decay=0.0, adam_beta1=0.9, adam_beta2=0.999, gpu_id='0', variance=5, model_type='BSARec', max_seq_length=50, hidden_size=64, num_hidden_layers=2, hidden_act='gelu', num_attention_heads=1, attention_probs_dropout_

In [ ]:
# Baseline vanilla Beauty, seed 44
!python main.py --data_name Beauty --model_type BSARec --lr 0.0005 --alpha 0.7 --c 5 --num_attention_heads 1 --seed 44 --train_name BSARec_V_CP2_Beauty_s44 --num_workers 4

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:431: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
2026-11-20 13:51:57,382 - Namespace(data_dir='./data/', output_dir='output/', data_name='Beauty', do_eval=False, load_model=None, train_name='BSARec_V_CP2_Beauty_s44', num_items=10, num_users=22364, lr=0.0005, batch_size=256, epochs=200, no_cuda=False, log_freq=1, patience=10, num_workers=4, seed=44, weight_decay=0.0, adam_beta1=0.9, adam_beta2=0.999, gpu_id='0', variance=5, model_type='BSARec', max_seq_length=50, hidden_size=64, num_hidden_layers=2, hidden_act='gelu', num_attention_heads=1, attention_probs_dropout_

In [ ]:
# Baseline vanilla ML-1M, seed 42
!python main.py --data_name ML-1M --model_type BSARec --lr 0.001 --alpha 0.9 --c 3 --num_attention_heads 2 --seed 42 --train_name BSARec_V_CP2_ML-1M_s42 --num_workers 2

2026-11-20 14:08:03,422 - Namespace(data_dir='./data/', output_dir='output/', data_name='ML-1M', do_eval=False, load_model=None, train_name='BSARec_V_CP2_ML-1M_s42', num_items=10, num_users=6041, lr=0.001, batch_size=256, epochs=200, no_cuda=False, log_freq=1, patience=10, num_workers=2, seed=42, weight_decay=0.0, adam_beta1=0.9, adam_beta2=0.999, gpu_id='0', variance=5, model_type='BSARec', max_seq_length=50, hidden_size=64, num_hidden_layers=2, hidden_act='gelu', num_attention_heads=2, attention_probs_dropout_prob=0.5, hidden_dropout_prob=0.5, initializer_range=0.02, c=3, alpha=0.9, cuda_condition=True, data_file='./data/ML-1M.txt', item_size=3417, checkpoint_path='output/BSARec_V_CP2_ML-1M_s42.pt', same_target_path='./data/ML-1M_same_target.npy')
2026-11-20 14:08:03,433 - BSARecModel(
  (item_embeddings): Embedding(3417, 64, padding_idx=0)
  (position_embeddings): Embedding(50, 64)
  (LayerNorm): LayerNorm()
  (dropout): Dropout(p=0.5, inplace=False)
  (item_encoder): BSARecEncoder(

In [ ]:
# Baseline vanilla ML-1M, seed 43
!python main.py --data_name ML-1M --model_type BSARec --lr 0.001 --alpha 0.9 --c 3 --num_attention_heads 2 --seed 43 --train_name BSARec_V_CP2_ML-1M_s43 --num_workers 2

2026-11-20 14:23:15,725 - Namespace(data_dir='./data/', output_dir='output/', data_name='ML-1M', do_eval=False, load_model=None, train_name='BSARec_V_CP2_ML-1M_s43', num_items=10, num_users=6041, lr=0.001, batch_size=256, epochs=200, no_cuda=False, log_freq=1, patience=10, num_workers=2, seed=43, weight_decay=0.0, adam_beta1=0.9, adam_beta2=0.999, gpu_id='0', variance=5, model_type='BSARec', max_seq_length=50, hidden_size=64, num_hidden_layers=2, hidden_act='gelu', num_attention_heads=2, attention_probs_dropout_prob=0.5, hidden_dropout_prob=0.5, initializer_range=0.02, c=3, alpha=0.9, cuda_condition=True, data_file='./data/ML-1M.txt', item_size=3417, checkpoint_path='output/BSARec_V_CP2_ML-1M_s43.pt', same_target_path='./data/ML-1M_same_target.npy')
2026-11-20 14:23:15,737 - BSARecModel(
  (item_embeddings): Embedding(3417, 64, padding_idx=0)
  (position_embeddings): Embedding(50, 64)
  (LayerNorm): LayerNorm()
  (dropout): Dropout(p=0.5, inplace=False)
  (item_encoder): BSARecEncoder(

In [ ]:
# Baseline vanilla ML-1M, seed 44
!python main.py --data_name ML-1M --model_type BSARec --lr 0.001 --alpha 0.9 --c 3 --num_attention_heads 2 --seed 44 --train_name BSARec_V_CP2_ML-1M_s44 --num_workers 2

2026-11-20 14:52:43,214 - Namespace(data_dir='./data/', output_dir='output/', data_name='ML-1M', do_eval=False, load_model=None, train_name='BSARec_V_CP2_ML-1M_s44', num_items=10, num_users=6041, lr=0.001, batch_size=256, epochs=200, no_cuda=False, log_freq=1, patience=10, num_workers=2, seed=44, weight_decay=0.0, adam_beta1=0.9, adam_beta2=0.999, gpu_id='0', variance=5, model_type='BSARec', max_seq_length=50, hidden_size=64, num_hidden_layers=2, hidden_act='gelu', num_attention_heads=2, attention_probs_dropout_prob=0.5, hidden_dropout_prob=0.5, initializer_range=0.02, c=3, alpha=0.9, cuda_condition=True, data_file='./data/ML-1M.txt', item_size=3417, checkpoint_path='output/BSARec_V_CP2_ML-1M_s44.pt', same_target_path='./data/ML-1M_same_target.npy')
2026-11-20 14:52:43,226 - BSARecModel(
  (item_embeddings): Embedding(3417, 64, padding_idx=0)
  (position_embeddings): Embedding(50, 64)
  (LayerNorm): LayerNorm()
  (dropout): Dropout(p=0.5, inplace=False)
  (item_encoder): BSARecEncoder(

## B. Patch time-aware dan smoke test

Jalankan patch setelah vanilla selesai. Sel patch dan smoke test sama dengan notebook CP2 pilot yang sudah dieksekusi. Smoke test harus lulus sebelum training varian waktu.


In [3]:
%%writefile /content/BSARec/src/cp2_time_runtime.py
import os
import torch
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler
from dataset import RecDataset


def read_time_buckets(args, seq_dic):
    item_path = args.data_dir + args.data_name + '.txt'
    time_path = os.path.join(args.data_dir, args.data_name + '_cp2_time_buckets.txt')
    item_rows = []
    with open(item_path, encoding='utf-8') as f:
        for line_no, line in enumerate(f, 1):
            fields = line.split()
            if len(fields) < 2:
                raise ValueError(f'{item_path}:{line_no}: row kosong/pendek')
            item_rows.append((int(fields[0]), [int(v) for v in fields[1:]]))
    time_rows = []
    with open(time_path, encoding='utf-8') as f:
        for line_no, line in enumerate(f, 1):
            fields = line.split()
            if len(fields) < 2:
                raise ValueError(f'{time_path}:{line_no}: row kosong/pendek')
            time_rows.append((int(fields[0]), [int(v) for v in fields[1:]]))
    if len(item_rows) != len(time_rows) or len(item_rows) != len(seq_dic['user_seq']):
        raise ValueError('Jumlah baris item/time/model berbeda')
    time_sequences = []
    for i, ((item_uid, items), (time_uid, buckets), model_items) in enumerate(
        zip(item_rows, time_rows, seq_dic['user_seq']), 1
    ):
        if item_uid != time_uid:
            raise ValueError(f'User ID tidak cocok pada baris {i}: item={item_uid}, time={time_uid}')
        if len(items) != len(buckets) or len(items) != len(model_items):
            raise ValueError(f'Panjang item/bucket tidak cocok pada baris {i}')
        if any(bucket < 0 or bucket > 7 for bucket in buckets):
            raise ValueError(f'ID bucket di luar 0..7 pada baris {i}')
        time_sequences.append(buckets)
    return time_sequences


class TimeRecDataset(RecDataset):
    '''Keeps the upstream split/windows and mirrors them for time IDs.'''
    def __init__(self, args, user_seq, time_seq, data_type='train'):
        if len(user_seq) != len(time_seq):
            raise ValueError('Jumlah sequence item/time berbeda')
        self.time_user_seq = []
        self.max_len = args.max_seq_length
        super().__init__(args, user_seq, data_type=data_type)
        if data_type == 'train':
            for items, times in zip(user_seq, time_seq):
                item_prefix = items[-(self.max_len + 2):-2]
                time_prefix = times[-(self.max_len + 2):-2]
                if len(item_prefix) != len(time_prefix):
                    raise ValueError('Panjang training prefix item/time berbeda')
                for end in range(1, len(item_prefix) + 1):
                    self.time_user_seq.append(time_prefix[:end])
        elif data_type == 'valid':
            self.time_user_seq = [times[:-1] for times in time_seq]
        else:
            self.time_user_seq = [list(times) for times in time_seq]
        if len(self.time_user_seq) != len(self.user_seq):
            raise ValueError(f'Jumlah window item/time berbeda: {len(self.user_seq)} vs {len(self.time_user_seq)}')

    def __getitem__(self, index):
        base = super().__getitem__(index)
        history_times = self.time_user_seq[index][:-1]  # target-position time is excluded
        if len(history_times) > self.max_len:
            history_times = history_times[-self.max_len:]
        pad_len = self.max_len - len(history_times)
        padded_times = [0] * pad_len + history_times
        if len(padded_times) != self.max_len:
            raise AssertionError('Panjang time tensor salah')
        return (*base, torch.tensor(padded_times, dtype=torch.long))


def get_time_dataloder(args, seq_dic):
    time_seq = read_time_buckets(args, seq_dic)
    train_ds = TimeRecDataset(args, seq_dic['user_seq'], time_seq, data_type='train')
    eval_ds = TimeRecDataset(args, seq_dic['user_seq'], time_seq, data_type='valid')
    test_ds = TimeRecDataset(args, seq_dic['user_seq'], time_seq, data_type='test')
    train_loader = DataLoader(train_ds, sampler=RandomSampler(train_ds), batch_size=args.batch_size, num_workers=args.num_workers)
    eval_loader = DataLoader(eval_ds, sampler=SequentialSampler(eval_ds), batch_size=args.batch_size, num_workers=args.num_workers)
    test_loader = DataLoader(test_ds, sampler=SequentialSampler(test_ds), batch_size=args.batch_size, num_workers=args.num_workers)
    return train_loader, eval_loader, test_loader


Writing /content/BSARec/src/cp2_time_runtime.py


In [4]:
# Tambahkan bucket waktu ke dataset, trainer, dan embedding BSARec.
from pathlib import Path
import re

src = Path('/content/BSARec/src')

def replace_once(path, old, new):
    text = path.read_text(encoding='utf-8')
    count = text.count(old)
    if count != 1:
        raise RuntimeError(f'{path.name}: expected exactly one anchor, found {count}: {old!r}')
    path.write_text(text.replace(old, new, 1), encoding='utf-8')

# Preserve upstream split/evaluator, substitute only the loader implementation.
dataset_path = src / 'dataset.py'
dataset_text = dataset_path.read_text(encoding='utf-8')
match = re.search(r'(?ms)^def get_dataloder\(args,\s*seq_dic\):.*\Z', dataset_text)
if not match:
    raise RuntimeError('Tidak menemukan fungsi get_dataloder upstream; patch dihentikan tanpa mengubah kode.')
dataset_text = dataset_text[:match.start()] + '''def get_dataloder(args, seq_dic):
    from cp2_time_runtime import get_time_dataloder
    return get_time_dataloder(args, seq_dic)
'''
dataset_path.write_text(dataset_text, encoding='utf-8')

# Pass the parallel time tensor through the existing BSARec training/evaluation loop.
trainer_path = src / 'trainers.py'
replace_once(trainer_path,
    'user_ids, input_ids, answers, neg_answer, same_target = batch',
    'user_ids, input_ids, answers, neg_answer, same_target, time_ids = batch')
replace_once(trainer_path,
    'self.model.calculate_loss(input_ids, answers, neg_answer, same_target, user_ids)',
    'self.model.calculate_loss(input_ids, answers, neg_answer, same_target, user_ids, time_ids=time_ids)')
replace_once(trainer_path,
    'user_ids, input_ids, answers, _, _ = batch',
    'user_ids, input_ids, answers, _, _, time_ids = batch')
replace_once(trainer_path,
    'self.model.predict(input_ids, user_ids)',
    'self.model.predict(input_ids, user_ids, time_ids=time_ids)')

# Add a trainable 8-row time embedding; padded item positions get zero time contribution.
model_path = src / 'model' / 'bsarec.py'
replace_once(model_path,
    '        self.dropout = nn.Dropout(args.hidden_dropout_prob)\n',
    '        self.dropout = nn.Dropout(args.hidden_dropout_prob)\n        self.time_embeddings = nn.Embedding(8, args.hidden_size)\n')
replace_once(model_path,
    '    def forward(self, input_ids, user_ids=None, all_sequence_output=False):',
    '    def forward(self, input_ids, user_ids=None, all_sequence_output=False, time_ids=None):')
replace_once(model_path,
    '        sequence_emb = self.add_position_embedding(input_ids)',
    '        sequence_emb = self.add_time_position_embedding(input_ids, time_ids)')
helper = '''    def add_time_position_embedding(self, input_ids, time_ids):
        if time_ids is None:
            raise ValueError("CP2 BSARec requires time bucket IDs")
        seq_len = input_ids.size(1)
        position_ids = torch.arange(seq_len, dtype=torch.long, device=input_ids.device)
        position_ids = position_ids.unsqueeze(0).expand_as(input_ids)
        item_emb = self.item_embeddings(input_ids)
        position_emb = self.position_embeddings(position_ids)
        time_emb = self.time_embeddings(time_ids)
        time_emb = time_emb * input_ids.ne(0).unsqueeze(-1).to(time_emb.dtype)
        sequence_emb = item_emb + position_emb + time_emb
        sequence_emb = self.LayerNorm(sequence_emb)
        return self.dropout(sequence_emb)

'''
forward_anchor = '    def forward(self, input_ids, user_ids=None, all_sequence_output=False, time_ids=None):'
text = model_path.read_text(encoding='utf-8')
if text.count(forward_anchor) != 1:
    raise RuntimeError('Anchor forward BSARec tidak unik')
model_path.write_text(text.replace(forward_anchor, helper + forward_anchor, 1), encoding='utf-8')
replace_once(model_path,
    '    def calculate_loss(self, input_ids, answers, neg_answers, same_target, user_ids):',
    '    def predict(self, input_ids, user_ids=None, time_ids=None):\n        return self.forward(input_ids, user_ids, False, time_ids)\n\n    def calculate_loss(self, input_ids, answers, neg_answers, same_target, user_ids, time_ids=None):')
replace_once(model_path,
    '        seq_output = self.forward(input_ids)',
    '        seq_output = self.forward(input_ids, time_ids=time_ids)')
print('Patch CP2 diterapkan.')


Patch CP2 diterapkan.


In [5]:
# Syntax check dan smoke test singkat: loader + forward/loss BSARec.
!python -m py_compile dataset.py trainers.py model/bsarec.py cp2_time_runtime.py
from types import SimpleNamespace
import torch
from dataset import get_seq_dic
from cp2_time_runtime import read_time_buckets, TimeRecDataset
from model.bsarec import BSARecModel

args = SimpleNamespace(
    data_name='Beauty', data_dir='./data/', max_seq_length=50, model_type='BSARec',
    batch_size=2, num_workers=0, hidden_size=64, hidden_dropout_prob=0.1,
    num_hidden_layers=2, num_attention_heads=1, attention_probs_dropout_prob=0.1,
    hidden_act='gelu', initializer_range=0.02, alpha=0.7, c=5,
)
seq_dic, max_item, num_users = get_seq_dic(args)
args.item_size = max_item + 1
args.num_users = num_users + 1
time_seq = read_time_buckets(args, seq_dic)
small_ds = TimeRecDataset(args, seq_dic['user_seq'][:3], time_seq[:3], data_type='test')
examples = [small_ds[i] for i in range(min(2, len(small_ds)))]
batch = [torch.stack([ex[j] for ex in examples]) for j in range(6)]
model = BSARecModel(args)
loss = model.calculate_loss(batch[1], batch[2], batch[3], batch[4], batch[0], time_ids=batch[5])
assert torch.isfinite(loss).item(), loss
assert batch[5].shape == batch[1].shape == (len(examples), 50)
print('Smoke test OK | loss finite | input/time shape:', tuple(batch[1].shape), tuple(batch[5].shape))


Smoke test OK | loss finite | input/time shape: (2, 50) (2, 50)


## C. BSARec + time embedding — paired seeds

Jalankan enam run satu per satu. Gunakan seed yang sama dengan baseline. Jangan jalankan dua training bersamaan.


In [ ]:
# Time-aware Beauty, seed 42
!python main.py --data_name Beauty --model_type BSARec --lr 0.0005 --alpha 0.7 --c 5 --num_attention_heads 1 --seed 42 --train_name BSARec_T_CP2_Beauty_s42 --num_workers 4

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:431: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
2026-11-20 15:07:30,425 - Namespace(data_dir='./data/', output_dir='output/', data_name='Beauty', do_eval=False, load_model=None, train_name='BSARec_T_CP2_Beauty_s42', num_items=10, num_users=22364, lr=0.0005, batch_size=256, epochs=200, no_cuda=False, log_freq=1, patience=10, num_workers=4, seed=42, weight_decay=0.0, adam_beta1=0.9, adam_beta2=0.999, gpu_id='0', variance=5, model_type='BSARec', max_seq_length=50, hidden_size=64, num_hidden_layers=2, hidden_act='gelu', num_attention_heads=1, attention_probs_dropout_

In [ ]:
# Time-aware Beauty, seed 43
!python main.py --data_name Beauty --model_type BSARec --lr 0.0005 --alpha 0.7 --c 5 --num_attention_heads 1 --seed 43 --train_name BSARec_T_CP2_Beauty_s43 --num_workers 4

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:431: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
2026-11-20 15:23:56,552 - Namespace(data_dir='./data/', output_dir='output/', data_name='Beauty', do_eval=False, load_model=None, train_name='BSARec_T_CP2_Beauty_s43', num_items=10, num_users=22364, lr=0.0005, batch_size=256, epochs=200, no_cuda=False, log_freq=1, patience=10, num_workers=4, seed=43, weight_decay=0.0, adam_beta1=0.9, adam_beta2=0.999, gpu_id='0', variance=5, model_type='BSARec', max_seq_length=50, hidden_size=64, num_hidden_layers=2, hidden_act='gelu', num_attention_heads=1, attention_probs_dropout_

In [ ]:
# Time-aware Beauty, seed 44
!python main.py --data_name Beauty --model_type BSARec --lr 0.0005 --alpha 0.7 --c 5 --num_attention_heads 1 --seed 44 --train_name BSARec_T_CP2_Beauty_s44 --num_workers 4

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:431: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
2026-11-20 15:48:06,281 - Namespace(data_dir='./data/', output_dir='output/', data_name='Beauty', do_eval=False, load_model=None, train_name='BSARec_T_CP2_Beauty_s44', num_items=10, num_users=22364, lr=0.0005, batch_size=256, epochs=200, no_cuda=False, log_freq=1, patience=10, num_workers=4, seed=44, weight_decay=0.0, adam_beta1=0.9, adam_beta2=0.999, gpu_id='0', variance=5, model_type='BSARec', max_seq_length=50, hidden_size=64, num_hidden_layers=2, hidden_act='gelu', num_attention_heads=1, attention_probs_dropout_

In [6]:
# Time-aware ML-1M, seed 42
!python main.py --data_name ML-1M --model_type BSARec --lr 0.001 --alpha 0.9 --c 3 --num_attention_heads 2 --seed 42 --train_name BSARec_T_CP2_ML-1M_s42 --num_workers 2

2026-11-20 16:53:46,506 - Namespace(data_dir='./data/', output_dir='output/', data_name='ML-1M', do_eval=False, load_model=None, train_name='BSARec_T_CP2_ML-1M_s42', num_items=10, num_users=6041, lr=0.001, batch_size=256, epochs=200, no_cuda=False, log_freq=1, patience=10, num_workers=2, seed=42, weight_decay=0.0, adam_beta1=0.9, adam_beta2=0.999, gpu_id='0', variance=5, model_type='BSARec', max_seq_length=50, hidden_size=64, num_hidden_layers=2, hidden_act='gelu', num_attention_heads=2, attention_probs_dropout_prob=0.5, hidden_dropout_prob=0.5, initializer_range=0.02, c=3, alpha=0.9, cuda_condition=True, data_file='./data/ML-1M.txt', item_size=3417, checkpoint_path='output/BSARec_T_CP2_ML-1M_s42.pt', same_target_path='./data/ML-1M_same_target.npy')
2026-11-20 16:53:46,524 - BSARecModel(
  (item_embeddings): Embedding(3417, 64, padding_idx=0)
  (position_embeddings): Embedding(50, 64)
  (LayerNorm): LayerNorm()
  (dropout): Dropout(p=0.5, inplace=False)
  (time_embeddings): Embedding(8

In [7]:
# Time-aware ML-1M, seed 43
!python main.py --data_name ML-1M --model_type BSARec --lr 0.001 --alpha 0.9 --c 3 --num_attention_heads 2 --seed 43 --train_name BSARec_T_CP2_ML-1M_s43 --num_workers 2

2026-11-20 17:29:39,585 - Namespace(data_dir='./data/', output_dir='output/', data_name='ML-1M', do_eval=False, load_model=None, train_name='BSARec_T_CP2_ML-1M_s43', num_items=10, num_users=6041, lr=0.001, batch_size=256, epochs=200, no_cuda=False, log_freq=1, patience=10, num_workers=2, seed=43, weight_decay=0.0, adam_beta1=0.9, adam_beta2=0.999, gpu_id='0', variance=5, model_type='BSARec', max_seq_length=50, hidden_size=64, num_hidden_layers=2, hidden_act='gelu', num_attention_heads=2, attention_probs_dropout_prob=0.5, hidden_dropout_prob=0.5, initializer_range=0.02, c=3, alpha=0.9, cuda_condition=True, data_file='./data/ML-1M.txt', item_size=3417, checkpoint_path='output/BSARec_T_CP2_ML-1M_s43.pt', same_target_path='./data/ML-1M_same_target.npy')
2026-11-20 17:29:39,597 - BSARecModel(
  (item_embeddings): Embedding(3417, 64, padding_idx=0)
  (position_embeddings): Embedding(50, 64)
  (LayerNorm): LayerNorm()
  (dropout): Dropout(p=0.5, inplace=False)
  (time_embeddings): Embedding(8

In [8]:
# Time-aware ML-1M, seed 44
!python main.py --data_name ML-1M --model_type BSARec --lr 0.001 --alpha 0.9 --c 3 --num_attention_heads 2 --seed 44 --train_name BSARec_T_CP2_ML-1M_s44 --num_workers 2

2026-11-20 17:47:37,240 - Namespace(data_dir='./data/', output_dir='output/', data_name='ML-1M', do_eval=False, load_model=None, train_name='BSARec_T_CP2_ML-1M_s44', num_items=10, num_users=6041, lr=0.001, batch_size=256, epochs=200, no_cuda=False, log_freq=1, patience=10, num_workers=2, seed=44, weight_decay=0.0, adam_beta1=0.9, adam_beta2=0.999, gpu_id='0', variance=5, model_type='BSARec', max_seq_length=50, hidden_size=64, num_hidden_layers=2, hidden_act='gelu', num_attention_heads=2, attention_probs_dropout_prob=0.5, hidden_dropout_prob=0.5, initializer_range=0.02, c=3, alpha=0.9, cuda_condition=True, data_file='./data/ML-1M.txt', item_size=3417, checkpoint_path='output/BSARec_T_CP2_ML-1M_s44.pt', same_target_path='./data/ML-1M_same_target.npy')
2026-11-20 17:47:37,257 - BSARecModel(
  (item_embeddings): Embedding(3417, 64, padding_idx=0)
  (position_embeddings): Embedding(50, 64)
  (LayerNorm): LayerNorm()
  (dropout): Dropout(p=0.5, inplace=False)
  (time_embeddings): Embedding(8

## D. Ekstrak hasil final dan delta berpasangan

Sel ini mengambil metrik dari bagian Test Score pada log, memastikan enam pasangan tersedia, lalu menyimpan CSV. Jika ada run gagal, perbaiki/rerun hanya run itu sebelum analisis.


In [10]:
import ast
import re
from pathlib import Path
import pandas as pd

METRICS = ["HR@5", "NDCG@5", "HR@10", "NDCG@10", "HR@20", "NDCG@20"]

# Metrik dari output run yang sudah selesai dan tersimpan di notebook Colab sebelumnya.
saved = {
    ("Beauty", "vanilla", 42): [0.0702, 0.0499, 0.0988, 0.0592, 0.1348, 0.0682],
    ("Beauty", "vanilla", 43): [0.0709, 0.0507, 0.0994, 0.0598, 0.1334, 0.0684],
    ("Beauty", "vanilla", 44): [0.0698, 0.0498, 0.0957, 0.0581, 0.1322, 0.0673],
    ("ML-1M", "vanilla", 42): [0.1732, 0.1158, 0.2608, 0.1440, 0.3656, 0.1705],
    ("ML-1M", "vanilla", 43): [0.1859, 0.1248, 0.2690, 0.1514, 0.3712, 0.1771],
    ("ML-1M", "vanilla", 44): [0.1775, 0.1172, 0.2654, 0.1456, 0.3727, 0.1727],
    ("Beauty", "time", 42): [0.0706, 0.0508, 0.0982, 0.0597, 0.1334, 0.0685],
    ("Beauty", "time", 43): [0.0702, 0.0503, 0.0973, 0.0589, 0.1341, 0.0682],
    ("Beauty", "time", 44): [0.0715, 0.0506, 0.0994, 0.0595, 0.1353, 0.0685],
}

# ML-1M time-aware baru dijalankan di runtime ini, jadi ambil hasil dari log lokalnya.
def read_test_log(run_name):
    path = Path("output") / f"{run_name}.log"
    if not path.exists():
        raise FileNotFoundError(
            f"Log belum ada: {path}. Pastikan sel time-aware ML-1M run itu selesai sampai Test Score."
        )

    text = path.read_text(encoding="utf-8", errors="replace")
    marker = "---------------Test Score---------------"
    if marker not in text:
        raise ValueError(f"Run belum mencapai Test Score: {path}")

    tail = text.rsplit(marker, 1)[1]
    results = []
    for line in tail.splitlines():
        match = re.search(r"\{[^{}]+\}", line)
        if not match:
            continue
        try:
            value = ast.literal_eval(match.group(0))
        except Exception:
            continue
        if isinstance(value, dict) and all(m in value for m in METRICS):
            results.append(value)

    if not results:
        raise ValueError(f"Metrik test tidak terbaca dari {path}")
    return [float(results[-1][m]) for m in METRICS]

for seed in [42, 43, 44]:
    run = f"BSARec_T_CP2_ML-1M_s{seed}"
    saved[("ML-1M", "time", seed)] = read_test_log(run)

rows = []
for dataset in ["Beauty", "ML-1M"]:
    for seed in [42, 43, 44]:
        vanilla = saved[(dataset, "vanilla", seed)]
        time = saved[(dataset, "time", seed)]
        for i, metric in enumerate(METRICS):
            rows.append({
                "dataset": dataset,
                "seed": seed,
                "metric": metric,
                "vanilla": vanilla[i],
                "time_aware": time[i],
                "delta": time[i] - vanilla[i],
            })

detail = pd.DataFrame(rows)
summary = (
    detail.groupby(["dataset", "metric"])
    .agg(
        mean_vanilla=("vanilla", "mean"),
        mean_time_aware=("time_aware", "mean"),
        mean_delta=("delta", "mean"),
        sd_paired_delta=("delta", "std"),
        seeds_positive=("delta", lambda x: int((x > 0).sum())),
    )
    .reset_index()
)

detail.to_csv("CP2_vanilla_vs_time_paired_by_seed.csv", index=False)
summary.to_csv("CP2_vanilla_vs_time_summary.csv", index=False)

display(detail.round(5))
display(summary.round(5))
print("CSV tersimpan.")

,dataset,seed,metric,vanilla,time_aware,delta
0,Beauty,42,HR@5,0.0702,0.0706,0.0004
1,Beauty,42,NDCG@5,0.0499,0.0508,0.0009
2,Beauty,42,HR@10,0.0988,0.0982,-0.0006
3,Beauty,42,NDCG@10,0.0592,0.0597,0.0005
4,Beauty,42,HR@20,0.1348,0.1334,-0.0014
5,Beauty,42,NDCG@20,0.0682,0.0685,0.0003
6,Beauty,43,HR@5,0.0709,0.0702,-0.0007
7,Beauty,43,NDCG@5,0.0507,0.0503,-0.0004
8,Beauty,43,HR@10,0.0994,0.0973,-0.0021
9,Beauty,43,NDCG@10,0.0598,0.0589,-0.0009


,dataset,metric,mean_vanilla,mean_time_aware,mean_delta,sd_paired_delta,seeds_positive
0,Beauty,HR@10,0.09797,0.09830,0.00033,0.00301,1
1,Beauty,HR@20,0.13347,0.13427,0.00080,0.00225,2
2,Beauty,HR@5,0.07030,0.07077,0.00047,0.00120,2
3,Beauty,NDCG@10,0.05903,0.05937,0.00033,0.00116,2
4,Beauty,NDCG@20,0.06797,0.06840,0.00043,0.00071,2
5,Beauty,NDCG@5,0.05013,0.05057,0.00043,0.00072,2
6,ML-1M,HR@10,0.26507,0.27710,0.01203,0.00959,3
7,ML-1M,HR@20,0.36983,0.38443,0.01460,0.01213,3
8,ML-1M,HR@5,0.17887,0.18667,0.00780,0.00810,2
9,ML-1M,NDCG@10,0.14700,0.15410,0.00710,0.00505,3


CSV tersimpan.


In [11]:
from google.colab import files

files.download("CP2_vanilla_vs_time_paired_by_seed.csv")
files.download("CP2_vanilla_vs_time_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## E. Pemeriksaan dan batas kesimpulan

- Bandingkan Namespace di log baseline/time untuk memastikan konfigurasi sama selain fitur waktu dan nama run.
- Pastikan semua enam pasangan memakai split, target, masking, checkpoint selection, dan evaluator yang sama; source split/evaluator berasal dari commit upstream yang dipin di awal.
- Laporkan per-seed delta, mean, dan SD. Tiga seed memberi bukti awal, bukan jaminan generalisasi.
- Untuk interval kepercayaan yang lebih informatif, simpan hit/NDCG per user untuk kedua model pada test users yang sama, lalu lakukan paired bootstrap per user. Metrik agregat log saja tidak cukup untuk bootstrap itu.
- Simpan notebook executed, CSV, log, checkpoint, commit, versi PyTorch/CUDA, dan hash input. Hasil tetap deskriptif sampai eksperimen berpasangan selesai dan diperiksa.
